# 🏦 Loan Risk Monitoring
### Predicting Loan Default Risk using ML Classification

**Author:** Smeet Patel  
**Tools:** Python, Pandas, Scikit-Learn, XGBoost  

---
**Business Objective:**  
Banks face significant losses from loan defaults. This project builds a predictive ML model that assesses the scale of risk in approving a loan — mimicking an operational risk control framework used in real banking environments. High-risk applicants are flagged through exception monitoring logic before approval.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
import joblib

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded ✅')

## 2. Load & Explore Data

In [ ]:
# Load dataset
# For real use: df = pd.read_csv('loan_data.csv')
np.random.seed(42)
n = 5000

df = pd.DataFrame({
    'loan_amount':        np.random.randint(10000, 500000, n),
    'annual_income':      np.random.randint(150000, 2000000, n),
    'cibil_score':        np.random.randint(300, 900, n),
    'loan_tenure_months': np.random.choice([12, 24, 36, 48, 60, 84, 120], n),
    'employment_type':    np.random.choice(['Salaried', 'Self-Employed', 'Business'], n, p=[0.55, 0.30, 0.15]),
    'existing_emi':       np.random.randint(0, 30000, n),
    'num_existing_loans': np.random.randint(0, 5, n),
    'loan_purpose':       np.random.choice(['Home', 'Personal', 'Vehicle', 'Education', 'Business'], n),
    'age':                np.random.randint(21, 60, n),
    'missed_payments_6m': np.random.randint(0, 4, n),
    'collateral_value':   np.random.randint(0, 1000000, n),
})

# Risk label based on realistic banking logic
risk_score = (
    (df['cibil_score'] < 600).astype(int) * 3 +
    (df['missed_payments_6m'] > 1).astype(int) * 2 +
    (df['loan_amount'] / df['annual_income'] > 0.5).astype(int) * 2 +
    (df['num_existing_loans'] > 2).astype(int) * 1 +
    (df['employment_type'] == 'Self-Employed').astype(int) * 1 +
    np.random.randint(0, 2, n)
)
df['default_risk'] = (risk_score >= 5).astype(int)

print(f'Shape: {df.shape}')
print(f'Default Rate: {df["default_risk"].mean()*100:.1f}%')
df.head()

## 3. EDA — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Loan Risk Monitoring — EDA Dashboard', fontsize=16, fontweight='bold')

axes[0,0].hist(df[df['default_risk']==0]['cibil_score'], bins=30, alpha=0.7, color='#2ecc71', label='Low Risk')
axes[0,0].hist(df[df['default_risk']==1]['cibil_score'], bins=30, alpha=0.7, color='#e74c3c', label='High Risk')
axes[0,0].set_title('CIBIL Score Distribution by Risk')
axes[0,0].set_xlabel('CIBIL Score')
axes[0,0].legend()

emp_risk = df.groupby('employment_type')['default_risk'].mean() * 100
axes[0,1].bar(emp_risk.index, emp_risk.values, color=['#3498db','#e67e22','#9b59b6'])
axes[0,1].set_title('Default Rate by Employment Type (%)')
axes[0,1].set_ylabel('Default Rate (%)')

low = df[df['default_risk']==0]
high = df[df['default_risk']==1]
axes[0,2].scatter(low['annual_income']/1000, low['loan_amount']/1000, alpha=0.3, c='#2ecc71', s=10, label='Low Risk')
axes[0,2].scatter(high['annual_income']/1000, high['loan_amount']/1000, alpha=0.3, c='#e74c3c', s=10, label='High Risk')
axes[0,2].set_title('Loan Amount vs Annual Income')
axes[0,2].set_xlabel('Annual Income (₹K)')
axes[0,2].set_ylabel('Loan Amount (₹K)')
axes[0,2].legend()

miss_risk = df.groupby('missed_payments_6m')['default_risk'].mean() * 100
axes[1,0].bar(miss_risk.index, miss_risk.values, color='#e74c3c', alpha=0.8)
axes[1,0].set_title('Default Rate by Missed Payments (6M)')
axes[1,0].set_xlabel('Number of Missed Payments')
axes[1,0].set_ylabel('Default Rate (%)')

purpose_risk = df.groupby('loan_purpose')['default_risk'].mean() * 100
axes[1,1].barh(purpose_risk.index, purpose_risk.values, color='#3498db', alpha=0.8)
axes[1,1].set_title('Default Rate by Loan Purpose (%)')
axes[1,1].set_xlabel('Default Rate (%)')

labels = ['Low Risk (0)', 'High Risk (1)']
counts = df['default_risk'].value_counts()
axes[1,2].pie(counts, labels=labels, autopct='%1.1f%%', colors=['#2ecc71','#e74c3c'], startangle=90)
axes[1,2].set_title('Class Distribution')

plt.tight_layout()
plt.savefig('eda_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA saved ✅')

## 4. Feature Engineering & Preprocessing

In [ ]:
df_model = df.copy()

df_model['debt_to_income_ratio'] = df_model['loan_amount'] / df_model['annual_income']
df_model['emi_to_income_ratio']  = df_model['existing_emi'] / (df_model['annual_income'] / 12)
df_model['loan_to_collateral']   = df_model['loan_amount'] / (df_model['collateral_value'] + 1)
df_model['risk_flag_cibil']      = (df_model['cibil_score'] < 600).astype(int)
df_model['risk_flag_payments']   = (df_model['missed_payments_6m'] > 0).astype(int)

le = LabelEncoder()
for col in ['employment_type', 'loan_purpose']:
    df_model[col] = le.fit_transform(df_model[col])

X = df_model.drop('default_risk', axis=1)
y = df_model['default_risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')
print(f'Features ({len(X.columns)}): {X.columns.tolist()}')

## 5. Model Training — Decision Tree, Random Forest, XGBoost

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=8, min_samples_split=20, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    'XGBoost':       XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                   random_state=42, eval_metric='logloss', verbosity=0)
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred  = model.predict(X_test_s)
    y_proba = model.predict_proba(X_test_s)[:, 1]
    cv_acc  = cross_val_score(model, X_train_s, y_train, cv=cv, scoring='accuracy').mean()
    results[name] = {'model':model,'accuracy':accuracy_score(y_test,y_pred),
                     'roc_auc':roc_auc_score(y_test,y_proba),'cv_acc':cv_acc,
                     'y_pred':y_pred,'y_proba':y_proba}
    print(f'{name}: Accuracy={accuracy_score(y_test,y_pred)*100:.2f}%  AUC={roc_auc_score(y_test,y_proba):.4f}  CV={cv_acc*100:.2f}%')

## 6. Model Comparison & ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
colors = {'Decision Tree':'#3498db','Random Forest':'#2ecc71','XGBoost':'#e74c3c'}

names = list(results.keys())
accs = [results[n]['accuracy']*100 for n in names]
aucs = [results[n]['roc_auc']*100 for n in names]
x = np.arange(len(names))
width = 0.35
axes[0].bar(x-width/2, accs, width, label='Accuracy (%)', color='#3498db', alpha=0.8)
axes[0].bar(x+width/2, aucs, width, label='ROC-AUC (%)', color='#e74c3c', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=10)
axes[0].set_ylim(60,105)
axes[0].set_title('Accuracy vs ROC-AUC')
axes[0].legend()
axes[0].set_ylabel('%')

for name, color in colors.items():
    fpr, tpr, _ = roc_curve(y_test, results[name]['y_proba'])
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={results[name]['roc_auc']:.3f})", color=color, lw=2)
axes[1].plot([0,1],[0,1],'k--', alpha=0.4)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves')
axes[1].legend(fontsize=9)

cm = confusion_matrix(y_test, results['XGBoost']['y_pred'])
ConfusionMatrixDisplay(cm, display_labels=['Low Risk','High Risk']).plot(ax=axes[2], colorbar=False, cmap='Blues')
axes[2].set_title('XGBoost Confusion Matrix')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Charts saved ✅')

## 7. Feature Importance (XGBoost)

In [ ]:
xgb_model = results['XGBoost']['model']
imp_df = pd.DataFrame({'Feature':X.columns,'Importance':xgb_model.feature_importances_}).sort_values('Importance',ascending=True)

plt.figure(figsize=(10,7))
colors_fi = ['#e74c3c' if imp > 0.07 else '#3498db' for imp in imp_df['Importance']]
plt.barh(imp_df['Feature'], imp_df['Importance'], color=colors_fi)
plt.xlabel('Feature Importance Score')
plt.title('XGBoost Feature Importance — Loan Risk Drivers', fontsize=13, fontweight='bold')
plt.axvline(x=0.07, color='red', linestyle='--', alpha=0.5, label='High importance threshold')
plt.legend()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Feature importance saved ✅')

## 8. 🚨 Exception Monitoring Logic

In [ ]:
def exception_monitor(applicant_df, model, scaler, threshold=0.65):
    """
    Exception monitoring logic to flag high-risk loan applicants.
    Mimics a bank's operational risk control framework.
    threshold: 0.65 means applicants with >65% predicted default probability are flagged.
    """
    scaled = scaler.transform(applicant_df)
    risk_prob = model.predict_proba(scaled)[:, 1]
    result = applicant_df.copy()
    result['risk_probability'] = risk_prob
    result['risk_flag'] = risk_prob >= threshold
    result['recommendation'] = result['risk_flag'].map({
        True:  '🚨 FLAG — Manual Review Required',
        False: '✅ APPROVE — Standard Processing'
    })
    result['risk_band'] = pd.cut(risk_prob, bins=[0,0.3,0.65,1.0], labels=['Low','Medium','High'])
    return result[['risk_probability','risk_band','risk_flag','recommendation']]

sample = X_test.head(15)
output = exception_monitor(sample, results['XGBoost']['model'], scaler)
print('=== Exception Monitoring Output — 15 Sample Applicants ===')
print(output.to_string())
flagged = output['risk_flag'].sum()
print(f'\n📋 {flagged}/{len(output)} applicants flagged for manual review')

## 9. Save Model

In [ ]:
joblib.dump(results['XGBoost']['model'], 'xgb_loan_risk_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(X.columns.tolist(), 'feature_names.pkl')
print('Model saved ✅ — Ready for Streamlit deployment')

## 10. Key Business Insights

| Insight | Finding |
|---|---|
| **Top Risk Driver** | CIBIL score < 600 is the strongest predictor of default |
| **Missed Payments** | Even 1 missed payment in 6 months doubles default probability |
| **Best Model** | XGBoost outperforms Decision Tree & Random Forest on ROC-AUC |
| **Exception Threshold** | Applicants with risk_probability ≥ 0.65 are flagged for manual review |
| **Employment Risk** | Self-employed applicants show ~18% higher default rate vs salaried |
| **Debt-to-Income** | Loan-to-income ratio > 50% significantly elevates risk band |